# 02. Track with MLflow

Moringa Masterclass: Machine Learning End to End

This is part 2 of 3. We repeat the training from notebook 1, but this time every run is logged with MLflow: parameters, metrics, and the trained model itself. Then we compare runs and register the better model as a version.

This is the part of the workflow that answers "which model is actually in production, and why" six months from now.

In [1]:
%pip install -q scikit-learn pandas numpy mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
!git clone https://github.com/BBWorksB/LLM_MaterClass
!ls

Cloning into 'LLM_MaterClass'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 4), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 179.16 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (4/4), done.
LLM_MaterClass	sample_data


In [4]:
%cd LLM_MaterClass

/content/LLM_MaterClass


In [5]:
import pandas as pd

df = pd.read_csv("data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

target = "Churn"
X = df.drop(columns=["customerID", target])
y = (df[target] == "Yes").astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 1. Point MLflow at a local tracking store

In a real production setup this would point at a shared MLflow tracking server. For the live session we use a local SQLite database, `mlflow.db`, so everyone can run this without any extra infrastructure. Recent MLflow versions no longer support a plain folder (`file:./mlruns`) as the tracking backend, so we use `sqlite:///mlflow.db` instead.

In [6]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("telco-churn")

2026/08/11 12:27:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/11 12:27:40 INFO mlflow.store.db.utils: Updating database tables
2026/08/11 12:27:44 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/LLM_MaterClass/mlruns/1', creation_time=1786451264280, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786451264280, lifecycle_stage='active', name='telco-churn', tags={}, trace_location=None, workspace='default'>

## 2. Wrap training in an MLflow run

Same pipeline as notebook 1. The only change is that we log parameters, metrics, and the fitted pipeline itself inside an `mlflow.start_run()` block.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

def train_and_log(run_name, classifier, params):
    with mlflow.start_run(run_name=run_name):
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ])
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        # mlflow.sklearn.log_model(pipeline, name="model")
        mlflow.sklearn.log_model(pipeline, name="model", serialization_format="pickle")

        print(run_name, metrics)
        return pipeline, metrics

In [11]:
log_reg_pipeline, log_reg_metrics = train_and_log(
    "logistic_regression",
    LogisticRegression(max_iter=1000, random_state=42),
    {"model_type": "logistic_regression", "max_iter": 1000},
)

2026/08/11 12:30:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


logistic_regression {'precision': 0.6572327044025157, 'recall': 0.5588235294117647, 'f1': 0.6040462427745664, 'roc_auc': np.float64(0.8421349040274871)}


In [12]:
rf_pipeline, rf_metrics = train_and_log(
    "random_forest",
    RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    {"model_type": "random_forest", "n_estimators": 200, "max_depth": 8},
)

2026/08/11 12:30:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest {'precision': 0.6821428571428572, 'recall': 0.5106951871657754, 'f1': 0.5840978593272171, 'roc_auc': np.float64(0.8414076829677853)}


## 3. Compare runs

We can pull runs back out of MLflow as a dataframe, which is often faster than opening the UI. Then we launch the actual MLflow UI to look at it visually.

In [13]:
runs = mlflow.search_runs(experiment_names=["telco-churn"])
runs[["tags.mlflow.runName", "metrics.f1", "metrics.roc_auc", "metrics.precision", "metrics.recall"]]

,tags.mlflow.runName,metrics.f1,metrics.roc_auc,metrics.precision,metrics.recall
0,random_forest,0.584098,0.841408,0.682143,0.510695
1,logistic_regression,0.604046,0.842135,0.657233,0.558824
2,random_forest,0.584098,0.841408,0.682143,0.510695
3,logistic_regression,0.604046,0.842135,0.657233,0.558824


In [18]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

https://5000-m-s-kkb-usw1c2-ayv39s7rp12d-c.us-west1-2.prod.colab.dev


To open the MLflow UI: in a terminal, from this folder, run

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

then open the URL it prints (usually http://127.0.0.1:5000). You will see both runs side by side, with their parameters, metrics, and the logged model artifact for each.

In Colab, the equivalent is to run the same command in a cell with `!`, then use a tunnel (e.g. `ngrok`) to view it, or simply rely on the `runs` dataframe above for the live session and demo the local UI from the presenter's machine.

## 4. Register the better model

Based on F1 and ROC-AUC, we register the stronger run as a named model version. This is the step that turns a one-off trained model into a versioned artifact other people and systems can reference by name.

In [16]:
best_run_name = "random_forest" if rf_metrics["f1"] >= log_reg_metrics["f1"] else "logistic_regression"
print("Registering:", best_run_name)

runs_sorted = mlflow.search_runs(
    experiment_names=["telco-churn"],
    filter_string=f"tags.mlflow.runName = '{best_run_name}'",
    order_by=["start_time DESC"],
)
best_run_id = runs_sorted.iloc[0]["run_id"]

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name="telco-churn-classifier")
print(registered)

Successfully registered model 'telco-churn-classifier'.
2026/08/11 12:36:56 WARNING mlflow.tracking._model_registry.fluent: Run with id 8a000844bc0843f0aa413ce17e43330f has no artifacts at artifact path 'model', registering model based on models:/m-5573d8194bea431ea0442bb068b8b333 instead


Registering: logistic_regression
<ModelVersion: aliases=[], creation_timestamp=1786451816720, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1786451816720, metrics=None, model_id=None, name='telco-churn-classifier', params=None, run_id='8a000844bc0843f0aa413ce17e43330f', run_link=None, source='models:/m-5573d8194bea431ea0442bb068b8b333', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>


Created version '1' of model 'telco-churn-classifier'.


## Recap

Both models are now logged with their parameters, metrics, and artifacts, and the better one is registered as `telco-churn-classifier`. If someone asks in six months which model is live and why it was chosen over the alternative, this is where the answer lives.

Next: `03_explain_shap_gemini.ipynb`, where we take individual predictions from the registered model, compute SHAP values, and use Gemini to turn those into plain-language explanations.